In [11]:
# Load env variables and create API client
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            return block.text

In [15]:
# Opcion recomendada ya que el curso usa stop_sequences que esta deprecado
from anthropic.types import ToolParam, tool_use_block

dataset_tool_schema = ToolParam({
    "name": "generate_dataset",
    "description": "Genera una lista de casos de prueba para evaluar el prompt.",
    "input_schema": {
        "type": "object",
        "properties": {
            "tasks": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"}
                    },
                    "required": ["task"]
                }
            }
        },
        "required": ["tasks"]
    }
})

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages,
        tools=[dataset_tool_schema],
        tool_choice={"type": "tool", "name": "generate_dataset"}
    )
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    return tool_use_block.input["tasks"]

In [16]:
dataset = generate_dataset()

print(dataset)


[{'task': 'Create a Python function that parses an AWS S3 bucket URI and returns the bucket name and object key separately'}, {'task': 'Write a JSON configuration object for an AWS Lambda function that defines environment variables for database connection (host, port, username)'}, {'task': 'Create a regex pattern that matches valid AWS IAM role ARNs in the format arn:aws:iam::account-id:role/role-name'}]


In [19]:
import json

dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
# Opcion alternativa usando re y json
# Explicacion de claude: https://claude.ai/share/2b9656fd-240c-46ff-80fc-787dda2e3db9
import json
import re

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
    """
    messages = []
    # El curso usa stop_sequences que esta deprecado
    add_user_message(messages, prompt)
    text = chat(messages)

    # Extrae el contenido entre ```json y ``` si existe
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if match:
        text = match.group(1)

    return json.loads(text.strip())